In [1]:
%pip install lightgbm xgboost catboost


  Using cached lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl (3.6 MB)
  Using cached xgboost-3.0.2-py3-none-manylinux_2_28_x86_64.whl (253.9 MB)
  Using cached catboost-1.2.8-cp310-cp310-manylinux2014_x86_64.whl (99.2 MB)
  Using cached nvidia_nccl_cu12-2.26.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (318.1 MB)
  Using cached plotly-6.1.2-py3-none-any.whl (16.3 MB)
  Using cached graphviz-0.20.3-py3-none-any.whl (47 kB)
  Using cached narwhals-1.41.0-py3-none-any.whl (357 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from src.config import *
from src.utils import get_latest_file
from src.predictions import *

from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
pd.set_option("display.max_columns", None)



In [8]:

# Paramètres
home_team_name = "Oklahoma City Thunder"
away_team_name = "Indiana Pacers"

home_odds = 1.22  # Cote de l'équipe à domicile
away_odds = 4.50  # Cote de l'équipe à l'extérieur

match_date = pd.to_datetime("2025-06-05")  # Date du match

# Chargement du dataset complet
dataset_path = get_latest_file(DATA_FINAL_DATASET_DIR)
full_df = pd.read_csv(dataset_path)
full_df["GAME_DATE"] = pd.to_datetime(full_df["GAME_DATE"])

# Mapping des noms d'équipes vers leurs IDs
team_mapping_file = get_latest_file(DATA_TEAMS_DIR)  # Fichier contenant les correspondances noms <-> IDs

team_mapping = pd.read_csv(team_mapping_file)

print("Mapping des équipes:")
print(team_mapping.head())


home_team_id = team_mapping.loc[team_mapping["full_name"] == home_team_name, "id"].values[0]
away_team_id = team_mapping.loc[team_mapping["full_name"] == away_team_name, "id"].values[0]






Mapping des équipes:
           id             full_name abbreviation         city          state  \
0  1610612737         Atlanta Hawks          ATL      Atlanta        Georgia   
1  1610612738        Boston Celtics          BOS       Boston  Massachusetts   
2  1610612739   Cleveland Cavaliers          CLE    Cleveland           Ohio   
3  1610612740  New Orleans Pelicans          NOP  New Orleans      Louisiana   
4  1610612741         Chicago Bulls          CHI      Chicago       Illinois   

   year_founded  
0          1949  
1          1946  
2          1970  
3          2002  
4          1966  


In [ ]:
# Filtrage du dataset pour inclure uniquement les données avant la date du match
cut_df = full_df[full_df["GAME_DATE"] < match_date].copy()

# Génération des features pour la prédiction
prediction_df = build_prediction_rows(home_team_id=home_team_id, away_team_id=away_team_id, dataset=cut_df, match_date=match_date)



prediction_df

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,MINUTES_PLAYED,OPP_TEAM_ID,OPP_GAME_DATE,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED,MATCHUP,SEASON,IS_HOME,IS_WIN,POINT_DIFF,ROLL_PTS_3,ROLL_PTS_5,ROLL_PTS_10,ROLL_PTS_25,ROLL_PTS_50,ROLL_PTS_100,ROLL_PTS_200,ROLL_REB_3,ROLL_REB_5,ROLL_REB_10,ROLL_REB_25,ROLL_REB_50,ROLL_REB_100,ROLL_REB_200,ROLL_AST_3,ROLL_AST_5,ROLL_AST_10,ROLL_AST_25,ROLL_AST_50,ROLL_AST_100,ROLL_AST_200,ROLL_FGM_3,ROLL_FGM_5,ROLL_FGM_10,ROLL_FGM_25,ROLL_FGM_50,ROLL_FGM_100,ROLL_FGM_200,ROLL_FGA_3,ROLL_FGA_5,ROLL_FGA_10,ROLL_FGA_25,ROLL_FGA_50,ROLL_FGA_100,ROLL_FGA_200,ROLL_FG_PCT_3,ROLL_FG_PCT_5,ROLL_FG_PCT_10,ROLL_FG_PCT_25,ROLL_FG_PCT_50,ROLL_FG_PCT_100,ROLL_FG_PCT_200,ROLL_PLUS_MINUS_3,ROLL_PLUS_MINUS_5,ROLL_PLUS_MINUS_10,ROLL_PLUS_MINUS_25,ROLL_PLUS_MINUS_50,ROLL_PLUS_MINUS_100,ROLL_PLUS_MINUS_200,ROLL_HOME_WINRATE_3,ROLL_AWAY_WINRATE_3,ROLL_HOME_WINRATE_5,ROLL_AWAY_WINRATE_5,ROLL_HOME_WINRATE_10,ROLL_AWAY_WINRATE_10,ROLL_HOME_WINRATE_25,ROLL_AWAY_WINRATE_25,ROLL_HOME_WINRATE_50,ROLL_AWAY_WINRATE_50,ROLL_HOME_WINRATE_100,ROLL_AWAY_WINRATE_100,ROLL_HOME_WINRATE_200,ROLL_AWAY_WINRATE_200,ROLL_WIN_RATIO_3,ROLL_WIN_RATIO_5,ROLL_WIN_RATIO_10,ROLL_WIN_RATIO_25,ROLL_WIN_RATIO_50,ROLL_WIN_RATIO_100,ROLL_WIN_RATIO_200,WIN_STREAK,HOME_WIN_STREAK,AWAY_WIN_STREAK,DAYS_SINCE_LAST_GAME,OPP_DAYS_SINCE_LAST_GAME,REST_ADVANTAGE,ROLL_HOME_REST_ADV_3,ROLL_AWAY_REST_ADV_3,ROLL_HOME_REST_ADV_5,ROLL_AWAY_REST_ADV_5,ROLL_HOME_REST_ADV_10,ROLL_AWAY_REST_ADV_10,ROLL_HOME_REST_ADV_25,ROLL_AWAY_REST_ADV_25,ROLL_HOME_REST_ADV_50,ROLL_AWAY_REST_ADV_50,ROLL_HOME_REST_ADV_100,ROLL_AWAY_REST_ADV_100,ROLL_HOME_REST_ADV_200,ROLL_AWAY_REST_ADV_200,ROLL_HOME_PTS_FOR_3,ROLL_HOME_OPP_PTS_AGAINST_3,ROLL_AWAY_PTS_FOR_3,ROLL_AWAY_OPP_PTS_AGAINST_3,ROLL_HOME_PTS_FOR_5,ROLL_HOME_OPP_PTS_AGAINST_5,ROLL_AWAY_PTS_FOR_5,ROLL_AWAY_OPP_PTS_AGAINST_5,ROLL_HOME_PTS_FOR_10,ROLL_HOME_OPP_PTS_AGAINST_10,ROLL_AWAY_PTS_FOR_10,ROLL_AWAY_OPP_PTS_AGAINST_10,ROLL_HOME_PTS_FOR_25,ROLL_HOME_OPP_PTS_AGAINST_25,ROLL_AWAY_PTS_FOR_25,ROLL_AWAY_OPP_PTS_AGAINST_25,ROLL_HOME_PTS_FOR_50,ROLL_HOME_OPP_PTS_AGAINST_50,ROLL_AWAY_PTS_FOR_50,ROLL_AWAY_OPP_PTS_AGAINST_50,ROLL_HOME_PTS_FOR_100,ROLL_HOME_OPP_PTS_AGAINST_100,ROLL_AWAY_PTS_FOR_100,ROLL_AWAY_OPP_PTS_AGAINST_100,ROLL_HOME_PTS_FOR_200,ROLL_HOME_OPP_PTS_AGAINST_200,ROLL_AWAY_PTS_FOR_200,ROLL_AWAY_OPP_PTS_AGAINST_200,H2H_LAST_3_DIFF,H2H_LAST_3_WINRATE,H2H_LAST_3_COUNT,H2H_LAST_5_DIFF,H2H_LAST_5_WINRATE,H2H_LAST_5_COUNT,H2H_LAST_10_DIFF,H2H_LAST_10_WINRATE,H2H_LAST_10_COUNT,H2H_LAST_25_DIFF,H2H_LAST_25_WINRATE,H2H_LAST_25_COUNT,H2H_LAST_50_DIFF,H2H_LAST_50_WINRATE,H2H_LAST_50_COUNT,H2H_LAST_100_DIFF,H2H_LAST_100_WINRATE,H2H_LAST_100_COUNT,H2H_LAST_200_DIFF,H2H_LAST_200_WINRATE,H2H_LAST_200_COUNT,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE,H2H_WIN_STREAK,ELO_PRE,OPP_ELO_PRE,ELO_PRE_SEASON,OPP_ELO_PRE_SEASON
64152,NaN,1610612750,2025-05-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1610612760,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-25,0,0,NaN,496.000000,416.4,313.6,265.68,250.84,234.82,247.17,176.000000,154.8,120.6,101.60,94.84,91.24,95.45,106.666667,92.8,71.8,60.08,57.60,53.34,56.91,178.666667,150.0,112.8,96.24,90.24,84.24,89.33,356.000000,311.2,240.2,203.12,191.32,181.08,187.82,21.126667,17.7200,12.5908,10.73224,9.82048,9.16688,10.14280,166.666667,18.0,41.0,64.8,68.4,45.0,56.20,0.5,0.0,0.666667,0.0,0.6,0.6,0.692308,0.583333,0.692308,0.62500,0.580000,0.580000,0.646465,0.594059,0.333333,0.4,0.6,0.64,0.66,0.58,0.62,0,1,0,2.0,2.0,0.0,0.0,0.0,0.000000,-39.5,-21.0,-15.8,-51.307692,-52.083333,-78.846154,-61.500000,-107.32000,-97.240000,-103.868687,-104.673267,538.0,458.0,412.0,472.0,439.333333,378.666667,382.0,464.0

In [ ]:
import joblib

#open last stacking model from DATA_MODELS_DIR 
stacking_model_path = get_latest_file(DATA_MODELS_DIR)
print("Loaded stacking model:", stacking_model_path)

# Charger le modèle de stacking
pipeline = joblib.load(stacking_model_path)


# Colonnes à ignorer
drop_cols = ['TEAM_ID','OPP_TEAM_ID','SEASON','GAME_DATE','IS_WIN']
X_pred = prediction_df.drop(columns=drop_cols, errors='ignore').drop(columns=COLS_MATCH_REAL, errors='ignore')

# ⚠️ Réordonner les colonnes si besoin
X_pred = X_pred[[col for col in pipeline.named_steps['scaler'].get_feature_names_out() if col in X_pred.columns]]

# Prédictions
pred_classes = pipeline.predict(X_pred)
pred_probas = pipeline.predict_proba(X_pred)[:, 1]

# Résultats
results_df = prediction_df[['TEAM_ID', 'IS_HOME']].copy()
results_df['PREDICTED_WIN'] = pred_classes
results_df['WIN_PROBA'] = pred_probas

# Affichage lisible avec nom d’équipe
results_df = results_df.merge(team_mapping[['id', 'full_name']], left_on='TEAM_ID', right_on='id', how='left')
results_df = results_df[['full_name', 'IS_HOME', 'PREDICTED_WIN', 'WIN_PROBA']].rename(columns={'full_name': 'TEAM'})

display(results_df)


# Affichage des résultats
for idx, team_id in enumerate(prediction_df["TEAM_ID"]):
    team_name = team_mapping.loc[team_mapping["id"] == team_id, "full_name"].values[0]
    print(f"{team_name}: {pred_probas[idx]*100:.2f}% de chances de gagner")


Loaded stacking model: data/models/stacking_model_2025-05-28_18-52-31.joblib


,TEAM,IS_HOME,PREDICTED_WIN,WIN_PROBA
0,Minnesota Timberwolves,0,0,0.345466
1,Oklahoma City Thunder,1,1,0.763709


Minnesota Timberwolves: 34.55% de chances de gagner
Oklahoma City Thunder: 76.37% de chances de gagner
